# Lumen3D — build a scene on a free Colab GPU

This notebook runs the **build** half of Lumen3D on a free Colab **T4**: upload a short
walkthrough video, and it reconstructs a language-queryable 3D scene (a `bundle.pkl` +
`scene.ply`). You can then query it right here, or download it and explore it locally
with `lumen3d view`.

**Before you start:** set the runtime to a GPU — *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi

## 1. Install

Install Lumen3D from GitHub plus the heavy models (Depth Anything 3, SAM 2, and SigLIP
via `transformers`). `addict` is a dependency DA3 forgets to declare, so we add it by hand.

> If the CUDA check in the next cell prints `False`, installing DA3 replaced Colab's GPU
> build of torch with a CPU one. Reinstall the CUDA build to match Colab's CUDA, e.g.
> `!pip install torch --index-url https://download.pytorch.org/whl/cu121`, then
> *Runtime → Restart runtime* and re-run from here.

In [ ]:
!git clone https://github.com/AwaisAdilKhokhar/lumen3d.git
%cd lumen3d
!pip install -q -e .
!pip install -q depth-anything-3 sam2 addict

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])   # DA3 needs <= 3.13
print("torch :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Provide a video

Upload a short (5–20 s) walkthrough clip — pan slowly around a room or an object so the
scene is seen from several angles. A phone clip is perfect.

In [ ]:
from google.colab import files

uploaded = files.upload()          # pick one .mp4
video = next(iter(uploaded))
print("uploaded:", video)

## 3. Build the scene

`reconstruct` runs the full pipeline — DA3 (depth) → SAM 2 (masks) → SigLIP (embeddings)
— and writes `myscene/bundle.pkl` + `myscene/scene.ply`.

**Fitting a free T4:** DA3's multi-view attention grows with the *square* of the frame
count, so frame count is the real memory lever. `--stride 60` (keep every 60th frame) plus
the smaller `DA3-BASE` backbone builds comfortably on a T4. Frames are also auto-downscaled
to 1024 px wide.

> **If you hit `CUDA out of memory`:** raise the stride (`--stride 90`), then add
> `--max-width 640`. Reconstruct runs as a subprocess, so an OOM crash frees the GPU — just
> re-run this cell with the new flags, no restart needed. For a denser, prettier cloud on a
> bigger GPU, lower the stride and switch to `depth-anything/DA3-LARGE`.

In [ ]:
!lumen3d reconstruct "{video}" -o myscene --stride 60 --model depth-anything/DA3-BASE

## 4. Query the scene

Turn a phrase into a ranked list of matching objects. Scores are low by SigLIP's design —
**rank, don't threshold.**

In [ ]:
!lumen3d query myscene "a wooden chair"

## 5. Preview the point cloud (optional)

A quick inline 3D scatter of the scene backdrop (subsampled so the browser stays snappy).

> **If this cell raises `TypeError: _reconstruct: First argument must be a sub-type of
> ndarray`:** the kernel started before the pip install, so it's holding an older NumPy than
> the one `reconstruct` used to pickle the scene. Fix: *Runtime → Restart session*, then run
> `%cd lumen3d` (a restart resets the working directory) and re-run this cell. No rebuild
> needed — `myscene/` is already on disk. The build and query cells are subprocesses, so
> they're immune; only this in-kernel cell is affected.

In [ ]:
import pickle, numpy as np
import plotly.graph_objects as go

bundle = pickle.load(open("myscene/bundle.pkl", "rb"))
points, colors = bundle["scene"]

n = min(20000, len(points))                       # subsample for responsiveness
idx = np.random.choice(len(points), n, replace=False)
p, c = points[idx], colors[idx]

fig = go.Figure(go.Scatter3d(
    x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers",
    marker=dict(size=1.5, color=[f"rgb({r},{g},{b})" for r, g, b in c]),
))
fig.update_layout(scene=dict(aspectmode="data"), margin=dict(l=0, r=0, t=0, b=0))
fig.show()

## 6. Download the scene

Zip the scene folder and download it. Locally (with Lumen3D installed) explore it in the
browser with:

```bash
lumen3d view myscene/
```

In [ ]:
!zip -qr myscene.zip myscene
from google.colab import files
files.download("myscene.zip")